In [ ]:
import pandas as pd
from collections import Counter
import re
import os
import warnings
import sys
import numpy as np
warnings.simplefilter('ignore')
sys.path.append("/2_findflags/all_subreddits")

In [ ]:
del sys.modules['functions_pos']

In [ ]:
import functions_pos as pos

In [ ]:
from IPython.core.display import display, HTML

def highlight_re(text, regex):
    htmld = re.sub(r'('+regex+')', '<mark>' + r'\1' + '</mark>', text.lower())
    display(HTML(htmld))
    
def print_submission(i, index, row):
    print('\n\n\nr/%s \n--------------------' % (index[0]))
    print('Published: %s' % row.subm_created[:20])
    print('%d | SUBM: %10s | COMM: %10s\n--------------------' % (i, index[1], index[2]))
    highlight_re('<b>'+row.subm_title+'</b>', covid_regex)
    print('URL: %s\nSubmission score: %d' % (row.subm_url, row.subm_score))

def print_comment(i, index, row, flag_no, limit = 1000):
    print('\nFLAG #%d (score = %d)' % (flag_no, row.comm_score))
    highlight_re(row.comm_body[:limit], fake_regex)    
    
def ftab(series, rnd = 3):
    return pd.concat([series.value_counts(),series.value_counts(normalize = True).round(rnd)],axis =1)

In [ ]:
united_file = sorted([filename for filename in os.listdir('../../output/') if re.search('UNITED_[0-9][0-9].+csv',filename)])[-1]
united_file

In [ ]:
COMSBSR = pd.read_csv('../../output/'+united_file).\
    assign(comm_direct = lambda df: df['comm_parent_id'].str.contains('t3_')).\
    set_index(['subreddit','subm_id','comm_id']).\
    sort_values('subm_created', ascending = False)
COMSBSR['comm_body'] = COMSBSR['comm_body'].astype(str)
COMSBSR.shape

In [ ]:
COMSBSR.to_csv('../../output/'+united_file+"_SRT")

In [ ]:
ftab(COMSBSR.comm_direct)

In [ ]:
covid_regex = '|'.join(pd.read_csv('../../input/keywords_covid.txt',header=None).loc[:,0].\
                  str.replace("\\(|\\)",r"").str.replace(r"\*",r""))#r"\\w*"))
fake_regex = '(fake |false).{0,15}(new\\w|info\\w|hood\\w)'+'|'+\
             '(dis|mis|mal)(info\\w|leading)'+'|'+\
             '\\bbs\\b|\\bbullshit\\b'+\
             '(not ).{0,5}(true\\b|verif\\w|real\\b)'+'|'+\
             '(unreli\\w+|unverifi\\w+|untrustw\\w+|fake\\w+)( source)'
fake_regex_no = '((no|not) .{0,10}('+fake_regex+'))'+'|(users often report submissions from this site)'

In [ ]:
highlight_re('some americans are misinformed about the coronavirus. how responsible is the media?', fake_regex)

In [ ]:
COMM_WITH_FLAGS = COMSBSR.\
    query('comm_body.str.contains("'+fake_regex+'")').\
    query('not comm_body.str.contains("'+fake_regex_no+'")').\
    query('not subm_title.str.contains("'+fake_regex+'")').\
    query('comm_direct')
COMM_WITH_FLAGS.shape

## Extract comments with POS

In [ ]:
fake_regex_partatt

In [ ]:
fake_regex_fullobj

In [ ]:
fake_regex_fullobj = '|'.join([r'\b'+w for w in pos.all_keywords['fullobjects']])
fake_regex_partobj = '|'.join([r'\b'+w for w in pos.all_keywords['partialobjects']])
fake_regex_partatt = '|'.join([r'\b'+w for w in pos.all_keywords['attributes']])

In [ ]:
pd.DataFrame({'b': TEMP.comm_body[10:20],
              'fo': TEMP.comm_body[10:20].str.contains(fake_regex_fullobj),
              'po': TEMP.comm_body[10:20].str.contains(fake_regex_partobj),
              'pa': TEMP.comm_body[10:20].str.contains(fake_regex_partatt)}).\
    assign(total = lambda df: df.fo | (df.po & df.pa))

In [ ]:
united_file = sorted([filename for filename in os.listdir('../../output/') if re.search('UNITED_[0-9][0-9].+csv',filename)])[-1]
READR = pd.read_csv('../../output/'+united_file, chunksize=100000)

In [ ]:
HASREGEX = pd.DataFrame()
n = 0
for TOP in READR:
    TEMP = TOP[(TOP.comm_body.str.contains(fake_regex_fullobj)) | \
               (TOP.comm_body.str.contains(fake_regex_partobj)) | \
               (TOP.comm_body.str.contains(fake_regex_partatt)) ]
    n+=TOP.shape[0]
    HASREGEX = HASREGEX.append(TEMP, ignore_index = True)
    print('Found %7d / %10d' % (HASREGEX.shape[0], n))

In [ ]:
HASREGEXSH = HASREGEX[\
               (HASREGEX.comm_body.str.contains(fake_regex_fullobj)) | \
               ((HASREGEX.comm_body.str.contains(fake_regex_partobj)) & \
                (HASREGEX.comm_body.str.contains(fake_regex_partatt))) ]
HASREGEXSH.shape

In [ ]:
HASREGEXSH.to_csv('../../output/HASREGEXSH.csv')

In [ ]:
HASREGEX.to_csv('../../output/HASREGEX.csv')

In [ ]:
fake_regex_pos = '|'.join(list({x for values in pos.all_keywords.values() for x in values}.difference(set(['it','be']))))
fake_regex_pos

In [ ]:
fake_regex_fullobj

In [ ]:
TOP.query(query_regex, engine='python')

### Printing

In [ ]:
def print_comm(DF):
    old_url, old_comm = 'blank','blank'
    iter_com, iter_found = 1,1
    for row in DF[DF['comm_direct']].itertuples():
        txt = row.comm_body
        url = row.subm_url
        if len(txt)<1000:
            for sent in pos.get_sent(txt):
                fp = sent._.fake_news_flag
                if fp:
                    #Print for each submission
                    if url!=old_url:
                        print('\n')
                        display(HTML('<i>r/'+ row.subreddit+'</i>'))
                        print(row.subm_url)
                        print(row.subm_title)
                        print(row.subm_created)
                        print('%4d / %6d' % (iter_found, iter_com))
                        old_url = url
                    #Print for each comment
                    if old_comm != txt:
                        highlight_re('<p>->: ' + txt + '</p>', fake_regex_pos)
                        old_comm = txt
                        iter_found+=1
                    #Print for each sentence
                    display(HTML('<p><b>'+fp+'</b></p>'))
    iter_com+=1

In [ ]:
print_comm(TOP)

In [ ]:
old_url, old_comm = 'blank','blank'
iter_com, iter_found = 1,1
for row in TOP[TOP['comm_direct']].itertuples():
    txt = row.comm_body
    url = row.subm_url
    if len(txt)<1000:
        for sent in pos.get_sent(txt):
            fp = sent._.fake_news_flag
            if fp:
                if url!=old_url:
                    print('\n')
                    display(HTML('<i>r/'+ row.subreddit+'</i>'))
                    print(url)
                    print(row.subm_created)
                    print('%4d / %6d' % (iter_found, iter_com))
                    old_url = url
                if old_comm != txt:
                    highlight_re('<p>->: ' + txt + '</p>', fake_regex_pos)
                    old_comm = txt
                    iter_found+=1
                display(HTML('<p><b>'+fp+'</b></p>'))
    iter_com+=1        

In [ ]:
870/188201

In [ ]:
sent_lst = []
for row in TOP[TOP['comm_direct']].itertuples():
    if len(row.comm_body)<1000:
        for sent in pos.get_sent(row.comm_body):
            sent_lst.append((row.comm_id, sent.text, sent._.fake_news_flag))

In [ ]:
sent_df = pd.DataFrame(sent_lst, columns = ['comm_id','sent','fake_news_flag'])
sent_df['comm_has_flags'] = sent_df.groupby('comm_id')['fake_news_flag'].transform('count')
sent_df['sent_id'] = sent_df.groupby('comm_id').cumcount() + 1
sent_df.set_index(['comm_id','sent_id'], inplace = True)
sent_df['fake_regex'] =np.logical_or(\
                        sent_df['sent'].str.contains(fake_regex_fullobj),
                        np.logical_and(sent_df['sent'].str.contains(fake_regex_partobj),
                                       sent_df['sent'].str.contains(fake_regex_partatt)))

In [ ]:
sent_df

### Comments not caught by the algo

In [ ]:
'is this website trustworthy?',
'misleading headline, thats not what he meant.',
'nice example of propaganda there',
'look this is extremely misleading',
'you call it wrongly attributed, i call it fake news',
'this is a propaganda article spewing more bs from beijing.',
'this sub is obviously propaganda',
'this sub looks like total propaganda trash',
'sorry this title is a bit misleading...',
"the title is misleading, it's talking about the government in general",
"the title of the article is sensationalized or misleading",
"clickbait title + paywall &#x200b; all it's missing the ads",
"that has to be one of the worst, most misleading headlines of any article on here.",
"completely nonsensical and misleading headline"

In [ ]:
for row in sent_df.itertuples():
    if row.fake_regex and row.fake_news_flag is None:
        highlight_re(row.sent, fake_regex_pos)
        print(row.fake_news_flag)
        print('\n')

In [ ]:
sent_df_bs = sent_df[sent_df['comm_has_flags']>0]
sent_df_bs['fake_news_flag']=sent_df_bs['fake_news_flag'].fillna('')
sent_df_bs = sent_df_bs[sent_df_bs['fake_news_flag'].str.lower().str.contains('bs|bullshit')]
for s,sent in zip(sent_df_bs.fake_news_flag, sent_df_bs.sent):
    print('%20s | %50s' % (s,sent))

In [ ]:
sent_df_bs.fake_news_flag.str.lower().value_counts()

In [ ]:
sent_df[~sent_df.fake_news_flag.isna()].fake_news_flag.str.lower().value_counts()[:50]